# Downstream classification experiment (data v2)

Task: predict `is_delayed` with XGBoost after missing values were injected into `total_freight_value` and imputed (Mean / Median / KNN / MICE), plus two baselines.

## What changed compared with the previous notebook

1. **Reads the v2 data only.** The old notebook loaded `exp_data/*.parquet`, which were generated from the *old* data. With the v2 `model_df` the order ids no longer match, and `isin()` would silently drop rows.
2. **Missing values are injected into the training split only; the test set stays clean** (as in the methodology: imputation is the experimental variable, the test set is the fixed yardstick). Before, the injection was applied to all rows and the test rows were imputed as well.
3. **Injection is repeated for every seed** (seed controls the split, the injection and the models). Before, one injected dataset (seed 42) was reused for all 5 seeds, so the injection randomness was never repeated.
4. Baseline label bug fixed (`"None"` vs `"Full_data"` made the Full-data comparison empty).
5. **Statistics: Friedman + Holm** instead of paired t-test / Wilcoxon on 5 values. With only 5 seeds an exact Wilcoxon signed-rank test can never get below p = 0.0625, so "not significant" would be guaranteed, not a finding.
6. `is_delayed` is cast to 0/1 integers; results are saved after every seed under new file names (old files are not overwritten).

Run with **Restart Kernel and Run All**.

## 0. Setup

In [1]:
import time
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer, KNNImputer
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from scipy.stats import friedmanchisquare, wilcoxon
from xgboost import XGBClassifier

# ---- paths -------------------------------------------------------------
BASE = Path("/Users/sivhoung/Desktop/Year 4 intern project/Project code ")
DATA_PATH = BASE / "dataset" / "processed dataset" / "final_df_(distance)_v2.parquet"
RESULTS_DIR = BASE / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SUFFIX = "datav2"
RAW_FILE = RESULTS_DIR / f"downstream_results_raw_{SUFFIX}.csv"
SUMMARY_FILE = RESULTS_DIR / f"downstream_results_summary_{SUFFIX}.csv"
DIFF_FILE = RESULTS_DIR / f"downstream_auc_diff_vs_full_{SUFFIX}.csv"
FRIEDMAN_FILE = RESULTS_DIR / f"downstream_friedman_by_scenario_{SUFFIX}.csv"
POSTHOC_FILE = RESULTS_DIR / f"downstream_pooled_posthoc_{SUFFIX}.csv"

# ---- experiment settings -----------------------------------------------
SEEDS = [42, 1, 7, 21, 100]
RATES = [0.10, 0.30, 0.50]
MECHANISMS = ["MCAR", "MAR", "MNAR"]
IMPUTERS = ["Mean", "Median", "KNN", "MICE"]
TARGET = "total_freight_value"        # variable that receives the missing values
AUXILIARY = "total_product_weight_g"  # drives the MAR mechanism
TEST_SIZE = 0.2

FEATURES = [
    "total_price",
    "total_freight_value",
    "total_product_weight_g",
    "average_product_length_cm",
    "average_product_height_cm",
    "item_count",
    "max_payment_installments",
    "average_distance_km",
]

## 1. Load data

In [2]:
df = pd.read_parquet(DATA_PATH)
print("full dataset:", df.shape)

model_df = df[["order_id", *FEATURES, "is_delayed"]].dropna().copy()
model_df["is_delayed"] = model_df["is_delayed"].astype(int)   # 0/1 integers for sklearn / XGBoost
model_df = model_df.reset_index(drop=True)

print("complete-case dataset:", model_df.shape)
print("missing values left:", int(model_df.isna().sum().sum()))
print(model_df["is_delayed"].value_counts())
print(model_df["is_delayed"].value_counts(normalize=True).round(4))

full dataset: (99418, 28)
complete-case dataset: (95959, 10)
missing values left: 0
is_delayed
0    89470
1     6489
Name: count, dtype: int64
is_delayed
0    0.9324
1    0.0676
Name: proportion, dtype: float64


## 2. Injection (training split only)

Same mechanisms as in the imputation notebook. The missing rate refers to the number of rows of the training split.

In [3]:
def inject_mcar(df, target, missing_rate, random_seed):
    out = df.copy()
    rng = np.random.default_rng(random_seed)
    n_missing = int(len(out) * missing_rate)
    idx = rng.choice(out.index, size=n_missing, replace=False)
    out.loc[idx, target] = np.nan
    return out


def inject_weighted(df, target, driver, missing_rate, random_seed):
    # higher value of `driver` -> higher probability of being set to missing
    out = df.copy()
    rng = np.random.default_rng(random_seed)
    w = out[driver].rank(pct=True).to_numpy() + 0.01
    w = w / w.sum()
    n_missing = int(len(out) * missing_rate)
    idx = rng.choice(out.index, size=n_missing, replace=False, p=w)
    out.loc[idx, target] = np.nan
    return out


def make_injected(train_df, mechanism, rate, seed):
    if mechanism == "MCAR":
        return inject_mcar(train_df, TARGET, rate, seed)
    if mechanism == "MAR":
        return inject_weighted(train_df, TARGET, AUXILIARY, rate, seed)
    if mechanism == "MNAR":
        return inject_weighted(train_df, TARGET, TARGET, rate, seed)
    raise ValueError(mechanism)

## 3. Imputation and model

In [4]:
def impute_train(train_df, method, seed):
    # Imputers are fitted on the training split only; the test split has no missing values.
    out = train_df.copy()

    if method == "Mean":
        out[TARGET] = out[TARGET].fillna(out[TARGET].mean())
    elif method == "Median":
        out[TARGET] = out[TARGET].fillna(out[TARGET].median())
    elif method in ("KNN", "MICE"):
        scaler = StandardScaler()
        X = scaler.fit_transform(out[FEATURES])
        if method == "KNN":
            imputer = KNNImputer(n_neighbors=5, weights="uniform")
        else:
            imputer = IterativeImputer(max_iter=10, random_state=seed)
        X = scaler.inverse_transform(imputer.fit_transform(X))
        out[FEATURES] = pd.DataFrame(X, columns=FEATURES, index=out.index)
    else:
        raise ValueError(method)

    assert out[FEATURES].notna().all().all(), f"{method}: missing values left"
    return out


def fit_and_evaluate(X_train, y_train, X_test, y_test, seed):
    scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

    model = XGBClassifier(
        n_estimators=100, max_depth=5, learning_rate=0.1,
        random_state=seed, eval_metric="logloss", n_jobs=-1,
        scale_pos_weight=scale_pos_weight,
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    return {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_prob),
    }

## 4. Run

For every seed: one stratified 80/20 split; two baselines on the clean training split (**Full_data**: no missing values; **Drop_column**: `total_freight_value` removed); then 3 mechanisms x 3 rates x 4 imputation methods on the injected training split. Everything is evaluated on the same clean test split. All methods of one seed share the split and the injected data, so they are paired.

Results are written after every seed. The first seed prints its run time, so you can estimate the total.

In [5]:
rows = []
t_start = time.time()

for seed in SEEDS:
    t_seed = time.time()

    train_df, test_df = train_test_split(
        model_df, test_size=TEST_SIZE, random_state=seed, stratify=model_df["is_delayed"]
    )
    X_test, y_test = test_df[FEATURES], test_df["is_delayed"]

    # baseline 1: full data (no missing values at all)
    metrics = fit_and_evaluate(train_df[FEATURES], train_df["is_delayed"], X_test, y_test, seed)
    rows.append({"seed": seed, "mechanism": "Baseline", "missing_rate": 0.0, "method": "Full_data", **metrics})

    # baseline 2: drop the column that would receive the missing values
    keep = [f for f in FEATURES if f != TARGET]
    metrics = fit_and_evaluate(train_df[keep], train_df["is_delayed"], test_df[keep], y_test, seed)
    rows.append({"seed": seed, "mechanism": "Baseline", "missing_rate": 0.0, "method": "Drop_column", **metrics})

    # 3 mechanisms x 3 rates x 4 imputation methods
    for mechanism in MECHANISMS:
        for rate in RATES:
            injected = make_injected(train_df, mechanism, rate, seed)
            n_missing = int(injected[TARGET].isna().sum())

            for method in IMPUTERS:
                imputed = impute_train(injected, method, seed)
                metrics = fit_and_evaluate(
                    imputed[FEATURES], imputed["is_delayed"], X_test, y_test, seed
                )
                rows.append({
                    "seed": seed, "mechanism": mechanism, "missing_rate": rate,
                    "method": method, "n_injected": n_missing, **metrics,
                })

    raw_df = pd.DataFrame(rows)
    raw_df.to_csv(RAW_FILE, index=False)
    print(
        f"seed {seed} done in {(time.time() - t_seed) / 60:.1f} min "
        f"(total {(time.time() - t_start) / 60:.1f} min), rows so far: {len(raw_df)}"
    )

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: in

seed 42 done in 4.5 min (total 4.5 min), rows so far: 38


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: in

seed 1 done in 4.6 min (total 9.1 min), rows so far: 76


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: in

seed 7 done in 4.6 min (total 13.6 min), rows so far: 114


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: in

seed 21 done in 4.6 min (total 18.2 min), rows so far: 152


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: in

seed 100 done in 6.2 min (total 24.5 min), rows so far: 190


In [6]:
expected = len(SEEDS) * (2 + len(MECHANISMS) * len(RATES) * len(IMPUTERS))
print("rows:", len(raw_df), "| expected:", expected)
print("NaN in metrics:", int(raw_df[["accuracy", "precision", "recall", "f1", "roc_auc"]].isna().sum().sum()))
raw_df.groupby("method")["seed"].nunique()

rows: 190 | expected: 190
NaN in metrics: 0


method
Drop_column    5
Full_data      5
KNN            5
MICE           5
Mean           5
Median         5
Name: seed, dtype: int64

## 5. Summary (mean and standard deviation over seeds)

In [7]:
summary = (
    raw_df
    .groupby(["mechanism", "missing_rate", "method"])
    .agg(
        auc_mean=("roc_auc", "mean"), auc_std=("roc_auc", "std"),
        f1_mean=("f1", "mean"), f1_std=("f1", "std"),
        recall_mean=("recall", "mean"), recall_std=("recall", "std"),
        precision_mean=("precision", "mean"), precision_std=("precision", "std"),
        accuracy_mean=("accuracy", "mean"), accuracy_std=("accuracy", "std"),
        n_runs=("seed", "nunique"),
    )
    .reset_index()
)
summary.to_csv(SUMMARY_FILE, index=False)

baselines = summary[summary["mechanism"] == "Baseline"][["method", "auc_mean", "auc_std", "f1_mean", "recall_mean"]]
baselines

,method,auc_mean,auc_std,f1_mean,recall_mean
0,Drop_column,0.604089,0.006680,0.158033,0.492296
1,Full_data,0.614494,0.006876,0.161314,0.521726


In [8]:
# ROC-AUC (mean over seeds): rows = method, columns = mechanism x missing rate
imputed_summary = summary[summary["mechanism"] != "Baseline"]
imputed_summary.pivot_table(
    index="method", columns=["mechanism", "missing_rate"], values="auc_mean"
).round(4)

mechanism        MAR                    MCAR                    MNAR          \
missing_rate     0.1     0.3     0.5     0.1     0.3     0.5     0.1     0.3   
method                                                                         
KNN           0.6119  0.6123  0.6067  0.6117  0.6108  0.6107  0.6128  0.6103   
MICE          0.6119  0.6105  0.6077  0.6129  0.6090  0.6090  0.6134  0.6109   
Mean          0.6120  0.6142  0.6087  0.6138  0.6120  0.6098  0.6126  0.6127   
Median        0.6131  0.6115  0.6086  0.6143  0.6101  0.6093  0.6133  0.6106   

mechanism             
missing_rate     0.5  
method                
KNN           0.6091  
MICE          0.6063  
Mean          0.6102  
Median        0.6094

In [9]:
# same, standard deviation over seeds
imputed_summary.pivot_table(
    index="method", columns=["mechanism", "missing_rate"], values="auc_std"
).round(4)

mechanism        MAR                    MCAR                    MNAR          \
missing_rate     0.1     0.3     0.5     0.1     0.3     0.5     0.1     0.3   
method                                                                         
KNN           0.0050  0.0041  0.0059  0.0043  0.0056  0.0045  0.0048  0.0062   
MICE          0.0052  0.0064  0.0069  0.0059  0.0051  0.0049  0.0036  0.0045   
Mean          0.0047  0.0050  0.0052  0.0047  0.0051  0.0049  0.0064  0.0037   
Median        0.0053  0.0067  0.0023  0.0053  0.0050  0.0051  0.0065  0.0060   

mechanism             
missing_rate     0.5  
method                
KNN           0.0058  
MICE          0.0051  
Mean          0.0067  
Median        0.0069

In [10]:
# Difference to the Full_data baseline, paired by seed (negative = worse than the complete data)
full = raw_df[raw_df["method"] == "Full_data"].set_index("seed")["roc_auc"]
drop = raw_df[raw_df["method"] == "Drop_column"].set_index("seed")["roc_auc"]

imp = raw_df[raw_df["mechanism"] != "Baseline"].copy()
imp["diff_vs_full"] = imp["roc_auc"].to_numpy() - imp["seed"].map(full).to_numpy()
imp["diff_vs_drop"] = imp["roc_auc"].to_numpy() - imp["seed"].map(drop).to_numpy()

diff = (
    imp.groupby(["mechanism", "missing_rate", "method"])
    .agg(
        diff_vs_full_mean=("diff_vs_full", "mean"), diff_vs_full_std=("diff_vs_full", "std"),
        diff_vs_drop_mean=("diff_vs_drop", "mean"), diff_vs_drop_std=("diff_vs_drop", "std"),
    )
    .reset_index()
)
diff.to_csv(DIFF_FILE, index=False)

print("Full_data AUC (mean over seeds):", round(full.mean(), 4), "| Drop_column AUC:", round(drop.mean(), 4))
diff.pivot_table(
    index="method", columns=["mechanism", "missing_rate"], values="diff_vs_full_mean"
).round(4)

Full_data AUC (mean over seeds): 0.6145 | Drop_column AUC: 0.6041


mechanism        MAR                    MCAR                    MNAR          \
missing_rate     0.1     0.3     0.5     0.1     0.3     0.5     0.1     0.3   
method                                                                         
KNN          -0.0026 -0.0022 -0.0078 -0.0028 -0.0037 -0.0037 -0.0017 -0.0042   
MICE         -0.0026 -0.0040 -0.0068 -0.0016 -0.0055 -0.0055 -0.0011 -0.0036   
Mean         -0.0025 -0.0003 -0.0058 -0.0007 -0.0025 -0.0047 -0.0019 -0.0018   
Median       -0.0014 -0.0029 -0.0059 -0.0002 -0.0044 -0.0052 -0.0012 -0.0039   

mechanism             
missing_rate     0.5  
method                
KNN          -0.0054  
MICE         -0.0082  
Mean         -0.0043  
Median       -0.0051

## 6. Statistical comparison of the imputation methods (ROC-AUC)

- **Friedman test** for the four imputation methods, blocks = seeds (paired: same split and same injected data within a seed).
- **Per scenario** (mechanism x rate) there are only 5 blocks, so the test has little power; p-values are Holm-adjusted over the 9 scenarios.
- **Pooled** over all scenarios: blocks = scenario x seed (45 blocks). The blocks are not fully independent (the same data and the same seeds appear in every scenario), so treat this as supporting evidence and say so in the report.
- Post-hoc after the pooled Friedman test: pairwise Wilcoxon signed-rank tests with Holm correction (6 pairs).

Read a non-significant result as "no difference detected", not as "the methods are equivalent".

In [11]:
def holm(pvals):
    p = np.asarray(pvals, dtype=float)
    m = len(p)
    order = np.argsort(p)
    adjusted = np.empty(m)
    running = 0.0
    for rank, idx in enumerate(order):
        running = max(running, (m - rank) * p[idx])
        adjusted[idx] = min(1.0, running)
    return adjusted


auc = (
    raw_df[raw_df["mechanism"] != "Baseline"]
    .pivot_table(index=["mechanism", "missing_rate", "seed"], columns="method", values="roc_auc")
    [IMPUTERS]
)

# --- per scenario ---------------------------------------------------------
records = []
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for (mech, rate), g in auc.groupby(level=["mechanism", "missing_rate"]):
        stat, p = friedmanchisquare(*[g[m].to_numpy() for m in IMPUTERS])
        mean_rank = g.rank(axis=1, ascending=False).mean()
        records.append({
            "mechanism": mech, "missing_rate": rate, "n_blocks": len(g),
            "friedman_chi2": stat, "p": p,
            "best_mean_rank": mean_rank.idxmin(),
            **{f"rank_{m}": mean_rank[m] for m in IMPUTERS},
        })

friedman_df = pd.DataFrame(records)
friedman_df["p_holm"] = holm(friedman_df["p"].fillna(1.0))
friedman_df.to_csv(FRIEDMAN_FILE, index=False)
friedman_df.round(4)

,mechanism,missing_rate,n_blocks,friedman_chi2,p,best_mean_rank,rank_Mean,rank_Median,rank_KNN,rank_MICE,p_holm
0,MAR,0.1,5,0.36,0.9484,Median,2.6,2.2,2.6,2.6,1.0000
1,MAR,0.3,5,4.92,0.1777,Mean,1.6,2.2,3.0,3.2,1.0000
2,MAR,0.5,5,3.96,0.2658,Mean,1.8,2.4,3.4,2.4,1.0000
3,MCAR,0.1,5,1.08,0.7819,Median,2.6,2.0,2.8,2.6,1.0000
4,MCAR,0.3,5,6.36,0.0954,Mean,1.6,2.6,2.2,3.6,0.8582
5,MCAR,0.5,5,0.36,0.9484,Mean,2.4,2.4,2.4,2.8,1.0000
6,MNAR,0.1,5,0.36,0.9484,MICE,2.6,2.6,2.6,2.2,1.0000
7,MNAR,0.3,5,1.08,0.7819,Mean,2.2,2.4,3.0,2.4,1.0000
8,MNAR,0.5,5,2.28,0.5164,Median,2.4,2.0,2.4,3.2,1.0000


In [12]:
# --- pooled over all scenarios ---------------------------------------------
stat, p = friedmanchisquare(*[auc[m].to_numpy() for m in IMPUTERS])
print(f"Pooled Friedman test ({len(auc)} blocks): chi2 = {stat:.3f}, p = {p:.4f}")
print("Mean rank (1 = best AUC):")
print(auc.rank(axis=1, ascending=False).mean().round(3).to_string())

pairs = [(a, b) for i, a in enumerate(IMPUTERS) for b in IMPUTERS[i + 1:]]
rows_ph = []
for a, b in pairs:
    d = auc[a].to_numpy() - auc[b].to_numpy()
    try:
        w_stat, w_p = wilcoxon(auc[a].to_numpy(), auc[b].to_numpy())
    except ValueError:  # all differences are zero
        w_stat, w_p = np.nan, 1.0
    rows_ph.append({
        "method_a": a, "method_b": b,
        "mean_auc_diff": d.mean(), "median_auc_diff": np.median(d),
        "share_a_better": (d > 0).mean(), "wilcoxon_p": w_p,
    })

posthoc = pd.DataFrame(rows_ph)
posthoc["p_holm"] = holm(posthoc["wilcoxon_p"])
posthoc.to_csv(POSTHOC_FILE, index=False)
posthoc.round(4)

Pooled Friedman test (45 blocks): chi2 = 6.680, p = 0.0828
Mean rank (1 = best AUC):
method
Mean      2.200
Median    2.311
KNN       2.711
MICE      2.778


,method_a,method_b,mean_auc_diff,median_auc_diff,share_a_better,wilcoxon_p,p_holm
0,Mean,Median,0.0006,0.0003,0.5333,0.2476,0.7428
1,Mean,KNN,0.0011,0.0007,0.6444,0.0426,0.2128
2,Mean,MICE,0.0016,0.0011,0.6222,0.0173,0.1037
3,Median,KNN,0.0005,0.0004,0.6000,0.4464,0.8929
4,Median,MICE,0.0010,0.0017,0.6222,0.1200,0.4801
5,KNN,MICE,0.0005,0.0005,0.5333,0.7038,0.8929


## Notes for the report

- All methods are evaluated on the **same clean test split**; missing values exist only in the training split. Say this in the method section.
- The injection, the split and the XGBoost seed are all controlled by the same seed, so the five repetitions are independent in all three respects.
- Compare the AUC differences with `Full_data` and `Drop_column`: if `Drop_column` is as good as `Full_data`, the freight feature carries little signal for delay prediction and no imputation method can show a large effect. That limits what the downstream task can tell, and should be stated openly.
- `n_injected` in the raw file is the number of values set to missing in the training split of that run.

file save to :
results/downstream_results_raw_datav2.csv
results/downstream_results_summary_datav2.csv
results/downstream_auc_diff_vs_full_datav2.csv
results/downstream_friedman_by_scenario_datav2.csv
results/downstream_pooled_posthoc_datav2.csv